In [1]:
import pandas as pd
import warnings
from sklearn.linear_model import RidgeClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.preprocessing import MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score

warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv("../data/processed/team_stats_processed.csv")
upcoming = pd.read_csv("../data/processed/upcoming_games.csv")
print(df.shape, upcoming.shape)
df.head()

(3230, 86) (544, 84)


,season,week,team,opponent_team,completions,attempts,passing_yards,passing_tds,passing_interceptions,sacks_suffered,...,opp_penalty_yards,opp_timeouts,opp_punt_returns,opp_punt_return_yards,opp_kickoff_returns,opp_kickoff_return_yards,opp_fg_made,opp_fg_att,opp_fg_long,opp_fg_pct
0,2020,1,ARI,SF,26,40,230,1,1,2,...,53,4,3,29,1,16,2,2,52.0,1.0
1,2020,1,ATL,SEA,37,54,450,2,1,2,...,46,4,1,15,2,43,1,1,42.0,1.0
2,2020,1,BAL,CLE,21,26,284,3,0,2,...,80,1,1,1,0,0,0,1,NaN,0.0
3,2020,1,BUF,NYJ,33,46,312,2,0,3,...,95,5,0,0,2,32,1,1,31.0,1.0
4,2020,1,CAR,LV,22,34,269,1,0,1,...,40,4,2,37,0,0,2,2,54.0,1.0


In [3]:
# Columns excluded from model features
NON_FEATURE = {
    "season", "week", "team", "opponent_team", "won",
    "overtime", "roof", "surface", "temp", "wind",  # unknowable pre-game or too sparse
}

FEATURE_COLS = [c for c in df.columns if c not in NON_FEATURE]
TARGET = "won"

print(f"{len(FEATURE_COLS)} features")
FEATURE_COLS

76 features


['completions',
 'attempts',
 'passing_yards',
 'passing_tds',
 'passing_interceptions',
 'sacks_suffered',
 'sack_yards_lost',
 'passing_air_yards',
 'passing_yards_after_catch',
 'passing_first_downs',
 'passing_epa',
 'passing_cpoe',
 'carries',
 'rushing_yards',
 'rushing_tds',
 'rushing_first_downs',
 'rushing_epa',
 'def_tackles_solo',
 'def_tackles_for_loss',
 'def_sacks',
 'def_qb_hits',
 'def_interceptions',
 'def_interception_yards',
 'def_pass_defended',
 'def_fumbles_forced',
 'penalties',
 'penalty_yards',
 'timeouts',
 'punt_returns',
 'punt_return_yards',
 'kickoff_returns',
 'kickoff_return_yards',
 'fg_made',
 'fg_att',
 'fg_long',
 'fg_pct',
 'div_game',
 'is_home',
 'rest',
 'opp_rest',
 'opp_completions',
 'opp_attempts',
 'opp_passing_yards',
 'opp_passing_tds',
 'opp_passing_interceptions',
 'opp_sacks_suffered',
 'opp_sack_yards_lost',
 'opp_passing_air_yards',
 'opp_passing_yards_after_catch',
 'opp_passing_first_downs',
 'opp_passing_epa',
 'opp_passing_cpoe',


In [4]:
# Impute and scale
imputer = SimpleImputer(strategy="mean")
scaler = MinMaxScaler()

X = pd.DataFrame(
    scaler.fit_transform(imputer.fit_transform(df[FEATURE_COLS])),
    columns=FEATURE_COLS,
    index=df.index
)
y = df[TARGET].astype(int)

X.shape

(3230, 76)

In [5]:
# Forward SFS with time-series CV to select 30 features
rr = RidgeClassifier(alpha=1)
split = TimeSeriesSplit(n_splits=3)
sfs = SequentialFeatureSelector(rr, n_features_to_select=30, direction="forward", cv=split)

sfs.fit(X, y)
predictors = list(X.columns[sfs.get_support()])
print(f"Selected {len(predictors)} predictors")
predictors

Selected 30 predictors


['passing_tds',
 'passing_epa',
 'rushing_tds',
 'rushing_first_downs',
 'rushing_epa',
 'def_interception_yards',
 'penalties',
 'punt_returns',
 'punt_return_yards',
 'kickoff_return_yards',
 'fg_made',
 'fg_att',
 'fg_long',
 'div_game',
 'is_home',
 'rest',
 'opp_passing_yards',
 'opp_sack_yards_lost',
 'opp_passing_air_yards',
 'opp_passing_epa',
 'opp_carries',
 'opp_rushing_yards',
 'opp_rushing_epa',
 'opp_def_interception_yards',
 'opp_penalties',
 'opp_penalty_yards',
 'opp_timeouts',
 'opp_fg_made',
 'opp_fg_att',
 'opp_fg_long']

In [6]:
# Cross-season backtest: train on all prior seasons, test on next
df_indexed = df.copy()
df_indexed[FEATURE_COLS] = X

seasons = sorted(df_indexed["season"].unique())
results = []

for i, test_season in enumerate(seasons[1:], start=1):
    train = df_indexed[df_indexed["season"] < test_season]
    test  = df_indexed[df_indexed["season"] == test_season]

    rr.fit(train[predictors], train[TARGET].astype(int))
    preds = rr.predict(test[predictors])

    acc = accuracy_score(test[TARGET].astype(int), preds)
    results.append({"test_season": test_season, "train_seasons": i, "accuracy": acc})
    print(f"{test_season}: {acc:.3f}  (trained on {i} season(s))")

results_df = pd.DataFrame(results)
print(f"Mean accuracy: {results_df["accuracy"].mean():.3f}")

2021: 0.919  (trained on 1 season(s))
2022: 0.878  (trained on 2 season(s))
2023: 0.941  (trained on 3 season(s))
2024: 0.930  (trained on 4 season(s))
2025: 0.925  (trained on 5 season(s))
Mean accuracy: 0.919


In [7]:
# Train final model on all available data
rr.fit(X[predictors], y)
print("Final model trained on all seasons")

Final model trained on all seasons


In [8]:
# Prepare upcoming game features using the fitted imputer + scaler
X_upcoming = pd.DataFrame(
    scaler.transform(imputer.transform(upcoming[FEATURE_COLS])),
    columns=FEATURE_COLS,
    index=upcoming.index
)

upcoming["predicted_win"] = rr.predict(X_upcoming[predictors])
upcoming[["season", "week", "team", "opponent_team", "is_home", "predicted_win"]]

,season,week,team,opponent_team,is_home,predicted_win
0,2026,1,SEA,NE,True,0
1,2026,1,LA,SF,True,1
2,2026,1,CAR,CHI,True,0
3,2026,1,CIN,TB,True,1
4,2026,1,DET,NO,True,1
...,...,...,...,...,...,...
539,2026,18,CHI,MIN,False,1
540,2026,18,MIA,NE,False,0
541,2026,18,TB,NO,False,1
542,2026,18,PHI,NYG,False,0


In [9]:
# One row per game (home team perspective) with predicted winner
home_view = upcoming[upcoming["is_home"] == True].copy()
home_view["predicted_winner"] = home_view.apply(
    lambda r: r["team"] if r["predicted_win"] == 1 else r["opponent_team"], axis=1
)

home_view[["season", "week", "team", "opponent_team", "predicted_winner"]].sort_values(["week", "team"])

,season,week,team,opponent_team,predicted_winner
2,2026,1,CAR,CHI,CHI
3,2026,1,CIN,TB,CIN
4,2026,1,DET,NO,DET
5,2026,1,HOU,BUF,BUF
6,2026,1,IND,BAL,IND
...,...,...,...,...,...
267,2026,18,MIN,CHI,CHI
268,2026,18,NE,MIA,NE
269,2026,18,NO,TB,TB
270,2026,18,NYG,PHI,PHI


In [10]:
matchups = home_view[["season", "week", "opponent_team", "team", "predicted_winner"]].copy()
matchups.columns = ["season", "week", "away_team", "home_team", "predicted_winner"]
matchups = matchups.sort_values(["week", "away_team"]).reset_index(drop=True)

matchups.to_csv("../data/processed/matchup_predictions.csv", index=False)
print(f"Saved {len(matchups)} matchups")
matchups

Saved 272 matchups


,season,week,away_team,home_team,predicted_winner
0,2026,1,ARI,LAC,LAC
1,2026,1,ATL,PIT,PIT
2,2026,1,BAL,IND,IND
3,2026,1,BUF,HOU,BUF
4,2026,1,CHI,CAR,CHI
...,...,...,...,...,...
267,2026,18,PIT,BAL,BAL
268,2026,18,SEA,LA,LA
269,2026,18,SF,ARI,SF
270,2026,18,TB,NO,TB


In [11]:
upcoming.to_csv("../data/processed/predictions.csv", index=False)
print(f"Saved {len(upcoming)} rows")

Saved 544 rows
